In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import polars as pl
from polars import col, lit, when
import re

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from scripts.feature_calculation import build_processed_dataset

При считывании создаем колонку, по которой можно отличить train от pretrain и приводим данные к одинаковой схеме

Пробуем на одной части для простоты

In [3]:
train = pl.scan_parquet('../../data/train_part_1.parquet')
train = train.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain = pl.scan_parquet('../../data/pretrain_part_1.parquet')
pretrain = pretrain.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)
labels = pl.scan_parquet('../../data/train_labels.parquet')

In [4]:
full_train = pl.concat([pretrain, train], how='vertical') 

In [5]:
build_processed_dataset(full_train)

  33,333 customers -> 50 partitions x ~667 customers each
[██████████████████████████████] 100.0%  part 50/50  (592,684 train rows)  elapsed 3s  ETA 0s             

Done. 31,101,860 total train rows written across 50 files in '../data_processed/'.


Посмотрим что получилось

In [6]:
example_data = pl.read_parquet('../data_processed/part_0000.parquet')
example_data.head()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,is_train,hour,day_of_week,day_of_month,week_of_year,hour_of_day,is_weekend,is_night,is_working_hour,minutes_from_midnight,log_amount,amount_abs,amount_round_100,amount_round_1000,…,global_event_desc_freq,circadian_deviation_score,channel_shift_score,velocity_change_flag,time_gap_variance_30d,new_device_flag,new_mcc_flag,new_channel_flag,merchant_entropy_user,browser_language_mismatch,new_device_and_night_flag,rdp_and_large_amount_flag,amount_zscore_given_channel,amount_zscore_given_mcc,amount_zscore_given_device,tx_time_zscore_given_user,amount_zscore_channel,amount_zscore_mcc,global_combination_freq,session_first_tx_flag,language_change_flag,os_change_flag,timezone_change_flag,rapid_sequence_flag,suspicious_env_flag,mcc_rare_global_flag,rare_combination_flag,device_change_and_large_amount_flag,voip_and_new_mcc_flag,compromised_and_high_amount_flag,session_first_tx_large_flag,geo_jump_proxy,device_entropy_ratio,language_mismatch,timezone_mismatch,merchant_last_seen_days,mcc_last_seen_days
i64,i64,datetime[μs],i32,i32,i32,i32,f32,i32,str,i32,str,str,i32,i64,i32,str,str,str,str,i32,i32,str,i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,f32,i8,i8,…,u32,f32,i8,i8,f32,i8,i8,i8,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,u32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i32,f32,i8,i8,i32,i32
123123123124290,123939168233221,2024-10-01 00:03:20,7,56,3,4,null,null,null,null,"""ru""",null,3,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,318,7.208777,0,0,1176.594116,0,0,0,0.0,0,0,0,991348.0625,1.1125e6,866895.5,4.166807,-0.066685,-0.036285,163343,0,0,0,0,0,0,1,0,0,0,0,0,0,0.018868,1,0,0,0
123123123126335,125107399907706,2024-10-01 00:07:13,14,75,6,5,22498.0,0,null,3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,7,10.021226,22498.0,0,0,…,925,8.977424,1,0,507.05838,0,0,0,0.0,0,0,0,373998.0625,1.1125e6,866895.5,5.576041,-0.128823,-0.035551,234595,0,0,0,0,0,0,1,0,0,0,0,0,0,0.004464,0,1,0,0
123123123127300,125605617099548,2024-10-01 00:09:29,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,9,0.0,0.0,1,1,…,667,10.206297,0,1,1488.790894,0,0,0,0.0,0,0,0,994301.6875,1.1125e6,866895.5,4.515576,-0.032788,-0.036285,632070,0,0,0,0,0,0,1,0,0,0,0,0,0,0.012346,0,1,0,0
123123123127300,125468178758568,2024-10-01 00:12:48,14,40,4,15,9962.0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,12,9.206634,9962.0,0,0,…,5,10.206297,0,1,1488.790894,0,0,0,0.0,0,0,0,994301.6875,1.1125e6,866895.5,4.515576,-0.032459,-0.03596,632070,0,0,0,0,1,0,1,0,0,0,0,0,0,0.012195,0,1,0,0
123123123126335,124961369601318,2024-10-01 00:14:55,14,75,6,5,17832.0,0,"""15""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,14,9.788806,17832.0,0,0,…,926,8.977424,0,0,507.05838,0,0,0,0.189829,0,0,0,373998.0625,66376.507812,866895.5,5.576041,-0.130533,-0.126332,234595,0,0,0,0,0,0,0,0,0,0,0,0,0,0.004444,0,1,0,0
